In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive (Recommended for saving heavy models)
drive.mount('/content/drive')

# 2. Configure Repo
# --- EDIT HERE ---
GITHUB_USERNAME = "lorenzo0-ctrl"     
REPO_NAME = "SemEval2026Task5"   
# -----------------

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 3. Clone 
if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone $REPO_URL
else:
    print("Repository already exists!")

# 4. Enter the directory
%cd $REPO_NAME

# 5. Install dependencies
print("Installing requirements...")
!pip install -r requirements.txt -q
print("Setup Completed!")

In [ ]:
import shutil

# Define where your train.json, dev.json, test.json files are located on Drive
# If you don't have them on Drive, upload them manually to the "Files" tab on the left and change path to "/content/"
SOURCE_PATH = "/content/drive/MyDrive/SemEval_Data" 
DEST_PATH = "data"

if not os.path.exists(DEST_PATH):
    os.makedirs(DEST_PATH)

required_files = ["train.json", "dev.json", "test.json"]

print(f"Looking for files in: {SOURCE_PATH}...")

for file_name in required_files:
    src = os.path.join(SOURCE_PATH, file_name)
    dst = os.path.join(DEST_PATH, file_name)
    
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied: {file_name}")
    elif os.path.exists(dst):
        print(f"Already present: {file_name}")
    else:
        print(f"MISSING: {file_name} (Please upload manually to {DEST_PATH})")

In [ ]:
# Training Parameters
OUTPUT_DIR = "outputs/mistral_qlora"

!PYTHONPATH=. python scripts/train.py \
    --data_dir data/ \
    --output_dir $OUTPUT_DIR \
    --epochs 8 \
    --batch_size 1 \
    --grad_accum 16 \
    --lr 5e-5 \
    --seed 42

print(f"Training finished! Model saved in {OUTPUT_DIR}")

In [ ]:
ADAPTER_PATH = "outputs/mistral_qlora"
SUBMISSION_FILE = "outputs/submission.jsonl"

print("Starting inference and calibration...")

!PYTHONPATH=. python scripts/predict.py \
    --adapter_path $ADAPTER_PATH \
    --dev_file data/dev.json \
    --test_file data/test.json \
    --output_file $SUBMISSION_FILE

print(f"File generated: {SUBMISSION_FILE}")

In [ ]:
from google.colab import files

if os.path.exists(SUBMISSION_FILE):
    files.download(SUBMISSION_FILE)
else:
    print("File not found. Did you run the previous cell?")